In [6]:
import time
import importlib.util
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

# Import your implementation from the same directory
spec = importlib.util.spec_from_file_location(
    "my_xgboost", "./XGBoost.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

XGBoostScratch = module.XGBoostScratch

# Famous real-world regression dataset
data = fetch_california_housing()

X = data.data
y = data.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset:", data.DESCR.splitlines()[0])
print("Features:", data.feature_names)
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)


Dataset: .. _california_housing_dataset:
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train shape: (16512, 8)
Validation shape: (4128, 8)


In [8]:

models = {
    "Scratch XGBoost": XGBoostScratch(),

    "sklearn GradientBoosting": GradientBoostingRegressor(
        n_estimators=module.n_estimators,
        learning_rate=module.lr,
        max_depth=module.max_depth,
        loss="squared_error",
        random_state=42
    ),

    "Library XGBoost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=module.n_estimators,
        learning_rate=module.lr,
        max_depth=module.max_depth,
        gamma=module.gamma,
        min_child_weight=module.min_child_weight,
        reg_lambda=module.Lambda,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )
}

results = []
predictions = {}

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - start

    start = time.perf_counter()
    train_pred = model.predict(X_train)
    train_predict_time = time.perf_counter() - start

    start = time.perf_counter()
    val_pred = model.predict(X_val)
    val_predict_time = time.perf_counter() - start

    predictions[name] = val_pred

    results.append({
        "Model": name,
        "Fit time (s)": fit_time,
        "Train predict (s)": train_predict_time,
        "Val predict (s)": val_predict_time,
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Val MSE": mean_squared_error(y_val, val_pred),
        "Val RMSE": np.sqrt(mean_squared_error(y_val, val_pred)),
        "Val MAE": mean_absolute_error(y_val, val_pred),
        "Train R2": r2_score(y_train, train_pred),
        "Val R2": r2_score(y_val, val_pred)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(5))

,Fit time (s),Train predict (s),Val predict (s),Train MSE,Val MSE,Val RMSE,Val MAE,Train R2,Val R2
Model,,,,,,,,,
Scratch XGBoost,45.67257,1.49063,0.37530,0.26160,0.29356,0.54181,0.37075,0.80431,0.77598
sklearn GradientBoosting,5.55569,0.01813,0.00394,0.26081,0.29400,0.54222,0.37164,0.80490,0.77564
Library XGBoost,2.30765,0.00638,0.00208,0.26128,0.29003,0.53855,0.36791,0.80454,0.77867
